# Data Preprocessing

## Creating Metadata

In [ ]:
import os
import pandas as pd
import numpy
import random
import librosa
import re
random.seed(42)
base_directory_yt = "audio_datasets/Youtube/diarization outputs filtered consolidated"
base_directory_torgo = "audio_datasets/TORGO"
output_metadata_csv = "audio_datasets/dataset_split.csv"
output_dataset_csv = "audio_datasets/dataset_all_audio.csv"
# os.makedirs(os.path.dirname(output_csv), exist_ok=True)
dataset_rows = []
metadata_rows = []



def extract_patient_code(name):
    # Match Torgo types that have "Session" in their name
    if "Session" in name:
        # Case 1: Name starts with patient ID like 'M01_Session...'
        return name.split("_")[0]
    else:
        # Match "audio 1_speaker 3" → extract 1 and 3
        match = re.search(r'audio\s*(\d+(?:\.\d+)?)_speaker\s*(\d+)', name, re.IGNORECASE)
        if match:
            audio_id = match.group(1)
            speaker_id = match.group(2)
            return f"A{audio_id}S{speaker_id}"
        print(f"Unknown Name : {name}")
        return "Unknown"



for audio_category in os.listdir(base_directory_yt):
  audio_cat_path = os.path.join(base_directory_yt,audio_category)
  
  for patient in os.listdir(audio_cat_path):

    speaker_id = f"{audio_category} {patient}"
    words = patient.lower().split()
    class_labels = {"normal":0,"mild":0,"mod":1, "moderate":1, "sev":2,"severe":2,"profound":3}
    severity = None
    for word in reversed(words):
      if word in class_labels:
        severity = word
        label = class_labels[word]
    
    if severity is None: 
      print(f"Audio : {audio_category}, Patient '{patient}' could not be categorized (unknown severity)")
      continue
    patient_path = os.path.join(audio_cat_path,patient)
    
    num_audio = 0
    for audio_length in os.listdir(patient_path):
      if audio_length == "segments _=2s": # >=2s became _=2s for some reason
        audio_length_path = os.path.join(patient_path, audio_length)

        for audio in os.listdir(audio_length_path):
          if not audio.endswith(".wav"):
            continue 
          file_path = os.path.join(audio_length_path, audio)
          try:
            waveform,sample_rate = librosa.load(file_path,sr=None)
          except Exception as e:
            print(f"Error reading {file_path}:{e}")
            continue
          num_audio +=1
          audio_path = os.path.abspath(file_path)
          name = f"{audio_category}_{patient}_{audio_length}_{audio.replace('.wav', '')}"
          patient_code = extract_patient_code(name)
          dataset_rows.append({"speaker_id":speaker_id, "audio_name":name,"label":label,"path":audio_path, "session_code":patient_code, "source": "youtube", "transcript":None})
      else:
        continue
    if num_audio == 0:
      print(f"Audio : {audio_category}, Patient '{patient}' has no audio files")
      continue
    metadata_rows.append({"speaker_id":speaker_id, "severity":severity, "label":label, "count":num_audio, "session_code":patient_code , "source": "youtube"})


togo_data = {
    "F3S1": {"severity": "Moderate", "label": 1},
    "F4S1": {"severity": "Mild", "label": 0},
    "M1S1": {"severity": "Profound", "label": 3},
    "M2S1": {"severity": "Severe", "label": 2},
    "M3S2": {"severity": "Mild", "label": 0},
    "M4S1": {"severity": "Profound", "label": 3},
    "M5S1": {"severity": "Severe", "label": 2},
    "F3S3": {"severity": "Severe", "label": 2},
    "F4S2": {"severity": "Mild", "label": 0},
    "M2S2": {"severity": "Profound", "label": 3},
    "M4S2": {"severity": "Profound", "label": 3},
    "M5S2": {"severity": "Moderate", "label": 1},
}


for patient, info in togo_data.items():
    severity = info["severity"]
    label = info["label"]
    # print(f"Patient: {patient}, Severity: {severity}, Label: {label}")
    gender = patient[0]  # Assuming first character
    num = patient[1]
    speaker_id = f"{gender}{int(num):02d}"
    session_num = patient[-1]
    session = f"Session{session_num}"
    # print(speaker_id)

    path = f"{base_directory_torgo}\\{speaker_id}\\{session}"
    if not os.path.exists(path):
        print(f"Path does not exist: {path}")
        continue
    
    session_path = os.path.join(path,"wav_headMic")
    if not os.path.exists(session_path):
        session_path = os.path.join(path,"wav_arrayMic")
        if not os.path.exists(session_path):
            print(f"Session path does not exist: {session_path}")
            continue

    audio_count = 0
    for filename in os.listdir(session_path):

        if not filename.endswith(".wav"):
            continue
        
        file_path = os.path.join(session_path, filename)
        
        try:
            waveform, sample_rate = librosa.load(file_path, sr=None)
        except Exception as e:
            print(f"Error reading {file_path}: {e}")
            continue

        txt_filename = filename.replace(".wav", ".txt")
        txt_file_path = os.path.join(path, "prompts", txt_filename)
        if not os.path.exists(txt_file_path):
            print(f"Prompt file does not exist: {txt_file_path}")
            continue

        with open(txt_file_path, 'r') as file:
            prompt = file.read().strip()

        if prompt.lower().endswith((".jpg", ".jpeg", "xxx")):
            continue
        if re.fullmatch(r"\[\s*[^]]+\s*\]", prompt):
            continue

        cleaned_prompt = re.sub(r"\[\s*[^]]+\s*\]", "", prompt).strip()
        if not cleaned_prompt:
            continue
        # Check that the prompt has more than 10 characters
        if len(cleaned_prompt) <= 10: 
            continue

        duration = len(waveform) / sample_rate
        if duration > 40: #more than 40s
            continue
        elif duration < 0.5: #less than 150ms skip
            continue
        
        audio_count += 1
        name = f"{patient}_{session}_{filename.replace('.wav', '')}"
        abs_path = os.path.abspath(file_path)
        text_path = os.path.abspath(txt_file_path)
        dataset_rows.append({"speaker_id":speaker_id, "audio_name":name,"label":label,"path":abs_path, "session_code":patient, "source": "TORGO", "transcript":cleaned_prompt})
    metadata_rows.append({"speaker_id":speaker_id, "severity":severity, "label":label, "count":audio_count, "session_code":patient, "source": "TORGO"})
  
     
df = pd.DataFrame(dataset_rows)
df.to_csv(output_dataset_csv,index=False)
meta = pd.DataFrame(metadata_rows)
meta.to_csv(output_metadata_csv,index=False)
display(df)
display(meta)

## Creating Datasplit

In [ ]:
import os
import pandas as pd
import numpy as np
import heapq
from math import floor, ceil

def summarize_splits_with_total(df: pd.DataFrame) -> pd.DataFrame:
    """
    Summarize train/val/test totals per numeric label and append a Combined All row.
    """
    required_cols = {"label", "split", "assigned_count"}
    if not required_cols.issubset(df.columns):
        raise ValueError(f"Missing columns: {required_cols - set(df.columns)}")

    # Per-label totals
    pivot = (
        df.groupby(["label", "split"])["assigned_count"]
        .sum()
        .unstack(fill_value=0)
        .reindex(columns=["train", "valid", "test"], fill_value=0)
    )
    pivot["total"] = pivot.sum(axis=1)
    pivot = pivot.reset_index().sort_values("label")

    # Rename columns with capitalized names
    pivot = pivot.rename(columns={
        "label": "Class",
        "train": "Train",
        "valid": "Val",
        "test": "Test",
        "total": "Total"
    })

    # Add Combined All row
    combined = pd.DataFrame([{
        "Class": "Combined All",
        "Train": pivot["Train"].sum(),
        "Val": pivot["Val"].sum(),
        "Test": pivot["Test"].sum(),
        "Total": pivot["Total"].sum()
    }])

    final_df = pd.concat([pivot, combined], ignore_index=True)
    return final_df


def split_dataset_by_patient(
    df: pd.DataFrame,
    save_path,
    target_class_column: str = "label",
    train_ratio: float = 0.8,
    valid_ratio: float = 0.1,
    test_ratio: float = 0.1,
    max_per_patient: int | None = None,   # absolute soft cap per patient; set None to disable
    class_slack: float = 0.20,            # legacy slack vs bottleneck
    dominance_cap: float = 1.25,          # cap each class to ≤ dominance_cap × average class total
    split_slack: float = 0.05,            # per-split slack around ratios
    min_allowed_per_patient: int = 1,     # never trim a patient below this
    random_state: int = 42,
) -> pd.DataFrame:
    """
    Split dataset by patient_code while:
      - Preserving EVERY patient (no row drops).
      - Preventing any class from dominating.
      - Reducing trimming via dominance_cap.
      - Assigning each patient to EXACTLY ONE split.
      - Hitting per-split ratios approximately using split_slack tolerance.

    Input columns: ['patient_code','count',target_class_column]
    Output columns: ['patient_code',target_class_column,'split','count','assigned_count']
    """
    if abs(train_ratio + valid_ratio + test_ratio - 1.0) > 1e-6:
        raise ValueError("Ratios must sum to 1.0")

    rng = np.random.RandomState(random_state)
    df_work = df.copy()

    # ---- Step 0: Optional soft-cap very large patients ----
    if max_per_patient is not None:
        df_work["assigned_count"] = df_work["count"].clip(upper=max_per_patient)
    else:
        df_work["assigned_count"] = df_work["count"].astype(int)

    # Ensure a floor for any nonzero original count
    nonzero_mask = df_work["count"] > 0
    df_work.loc[nonzero_mask, "assigned_count"] = (
        df_work.loc[nonzero_mask, "assigned_count"].clip(lower=min_allowed_per_patient)
    ).astype(int)

    # ---- Helper: one-by-one leveling ----
    def level_trim_one_by_one(class_df: pd.DataFrame, target_total: int) -> pd.Series:
        counts = class_df["assigned_count"].astype(int).copy()
        indices = list(class_df.index)
        current_total = int(counts.sum())
        if current_total <= target_total:
            return counts

        min_total = len(counts) * min_allowed_per_patient
        if target_total < min_total:
            target_total = min_total

        remaining = current_total - target_total
        heap = [(-int(counts.loc[i]), i) for i in indices]
        heapq.heapify(heap)
        cur = {i: int(counts.loc[i]) for i in indices}

        while remaining > 0 and heap:
            neg_c, i = heapq.heappop(heap)
            c = -neg_c
            if c <= min_allowed_per_patient:
                reducible_found = False
                buffer = []
                while heap:
                    neg_c2, j = heapq.heappop(heap)
                    c2 = -neg_c2
                    if c2 > min_allowed_per_patient:
                        c2 -= 1
                        remaining -= 1
                        cur[j] = c2
                        heapq.heappush(heap, (-c2, j))
                        for item in buffer:
                            if -item[0] > min_allowed_per_patient:
                                heapq.heappush(heap, item)
                        reducible_found = True
                        break
                    else:
                        buffer.append((neg_c2, j))
                if not reducible_found:
                    break
                continue

            c -= 1
            remaining -= 1
            cur[i] = c
            heapq.heappush(heap, (-c, i))

        out = counts.copy()
        for i, v in cur.items():
            out.loc[i] = max(v, min_allowed_per_patient)
        return out.astype(int)

    # ---- Step 1: Decide per-class targets ----
    classes = sorted(df_work[target_class_column].unique())
    class_totals = df_work.groupby(target_class_column)["assigned_count"].sum().to_dict()
    class_patients = df_work.groupby(target_class_column)["session_code"].nunique().to_dict()

    avg_class_total = int(round(sum(class_totals.values()) / max(1, len(classes))))
    bottleneck = min(class_totals.values())

    legacy_cap = {cls: max(class_patients[cls] * min_allowed_per_patient,
                           int((1.0 + class_slack) * bottleneck))
                  for cls in classes}

    dom_cap_val = int(dominance_cap * avg_class_total)

    targets_per_class = {}
    for cls in classes:
        min_feasible = class_patients[cls] * min_allowed_per_patient
        targets_per_class[cls] = min(
            class_totals[cls],
            max(min_feasible, min(legacy_cap[cls], dom_cap_val))
        )

    # ---- Step 2: Apply trimming ----
    trimmed_blocks = []
    for cls in classes:
        cls_df = df_work[df_work[target_class_column] == cls].copy()
        target_total = int(targets_per_class[cls])
        cls_df["assigned_count"] = level_trim_one_by_one(cls_df, target_total)
        trimmed_blocks.append(cls_df)

    df_trimmed = pd.concat(trimmed_blocks, axis=0)

    # ---- Step 3: Build per-split class targets ----
    post_totals = df_trimmed.groupby(target_class_column)["assigned_count"].sum().to_dict()
    split_names = ["train", "valid", "test"]
    split_ratios = {"train": train_ratio, "valid": valid_ratio, "test": test_ratio}

    def split_bounds(total: int):
        lows, highs = {}, {}
        for s in split_names:
            desired = total * split_ratios[s]
            lows[s] = int(floor(desired * (1 - split_slack)))
            highs[s] = int(ceil(desired * (1 + split_slack)))
        return lows, highs

    class_bounds = {cls: split_bounds(int(post_totals[cls])) for cls in classes}

    # ---- Step 4: Assign each patient ----
    result_rows = []
    for cls in classes:
        cls_df = df_trimmed[df_trimmed[target_class_column] == cls].copy()
        cls_df = cls_df.sample(frac=1.0, random_state=random_state + int(rng.randint(0, 1_000_000)))
        cls_df = cls_df.sort_values("assigned_count", ascending=False)

        lows, highs = class_bounds[cls]
        allocated = {s: 0 for s in split_names}

        for _, row in cls_df.iterrows():
            cnt = int(row["assigned_count"])
            need_low = {s: max(0, lows[s] - allocated[s]) for s in split_names}
            best_split = max(need_low, key=lambda s: need_low[s])
            if need_low[best_split] == 0:
                headroom = {s: max(0, highs[s] - allocated[s]) for s in split_names}
                best_split = max(headroom, key=lambda s: headroom[s])
                if headroom[best_split] == 0:
                    best_split = min(split_names, key=lambda s: allocated[s])

            result_rows.append({
                "speaker_id": row["speaker_id"],
                "session_code": row["session_code"],
                "source": row["source"],
                "severity": row["severity"],
                target_class_column: cls,
                "split": best_split,
                "count": int(row["count"]),
                "assigned_count": cnt,
            })
            allocated[best_split] += cnt
        result_df = pd.DataFrame(result_rows)
        def enforce_min_1_speaker_per_class_per_split(result_df, target_class_col="label",
                                                split_col="split",
                                                speaker_col="session_code",
                                                splits=("train","valid","test")):
            """
            Ensure each split has >=1 speaker for every class (if feasible).
            Moves the smallest-assigned_count speaker within a class from a donor split to a missing split.
            Ignores ratio rules. Preserves one-speaker-one-split.
            """
            df = result_df.copy()

            # Work class-by-class
            for cls, gcls in df.groupby(target_class_col):
                # speakers per split
                split_to_idxs = {s: gcls.index[gcls[split_col] == s].tolist() for s in splits}
                # Quick feasibility: if total speakers in class < number of splits, we can only fill up to that many splits
                total_speakers_cls = gcls[speaker_col].nunique()
                max_fillable_splits = min(len(splits), total_speakers_cls)

                # Recompute dynamically until stable or no more moves possible
                changed = True
                while changed:
                    changed = False

                    # Compute which splits lack presence (0 speakers)
                    missing = [s for s in splits if len(split_to_idxs[s]) == 0]

                    # If there are more missing splits than we can possibly fill, just try our best
                    if not missing:
                        break

                    for s_missing in list(missing):
                        # pick donor split: one with the MOST speakers (and at least 2 so it remains non-empty),
                        # otherwise the one with the largest cumulative assigned_count.
                        donor_candidates = [
                            (s, split_to_idxs[s]) for s in splits if len(split_to_idxs[s]) > 0
                        ]
                        if not donor_candidates:
                            # nothing to donate
                            continue

                        # Prefer donors that have >1 speakers so we don't create a new hole
                        donors_multi = [(s, idxs) for s, idxs in donor_candidates if len(idxs) > 1]
                        if donors_multi:
                            donor_pool = donors_multi
                        else:
                            donor_pool = donor_candidates  # may create a hole elsewhere, but requirement says enforce presence

                        # Choose donor by: max speaker count, tie-breaker by total assigned_count descending
                        def donor_key(pair):
                            s, idxs = pair
                            total_cnt = df.loc[idxs, "assigned_count"].sum()
                            return (len(idxs), total_cnt)

                        s_donor, donor_idxs = max(donor_pool, key=donor_key)

                        # Choose the SMALLEST patient in donor to minimize disruption
                        donor_row = df.loc[donor_idxs].sort_values("assigned_count", ascending=True).iloc[0]
                        donor_idx = donor_row.name

                        # Move it
                        df.at[donor_idx, split_col] = s_missing

                        # update bookkeeping for next iterations
                        split_to_idxs[s_donor].remove(donor_idx)
                        split_to_idxs[s_missing].append(donor_idx)
                        changed = True

                        # continue trying to fill other missing splits in this while-loop

            return df

    # Enforce the rule
    result_df = enforce_min_1_speaker_per_class_per_split(
        result_df,
        target_class_col="label",
        split_col="split",
        speaker_col="session_code",
        splits=("train","valid","test"),
    )

    
    summary_df = summarize_splits_with_total(result_df)

    # ---- Save outputs ----
    total_final = int(result_df["assigned_count"].sum())
    total_original = int(df_work["count"].sum())

    lines = []
    lines.append("=== Final Results (One-by-one trim with DOMINANCE + SPLIT slack; All Patients Preserved) ===")
    lines.append(f"Total samples (original):  {total_original:,}")
    lines.append(f"Total samples (final):     {total_final:,}")
    lines.append(f"Patients preserved:        {result_df['session_code'].nunique()} / {df['session_code'].nunique()}")
    lines.append(f"Bottleneck (pre-trim):     {bottleneck:,}")
    lines.append(f"Avg class total:           {avg_class_total:,}")
    lines.append(f"class_slack: {class_slack:.0%}; dominance_cap: {dominance_cap:.0%}; split_slack: {split_slack:.0%}")

    summary_text = summary_df.to_string(index=False)
    content = "\n".join(lines) + "\n\n" + summary_text

    save_summary_path = os.path.join(save_path,"summary.txt")
    save_csv_path = os.path.join(save_path,"metadata.csv")
    result_df.to_csv(save_csv_path,index=False)

    os.makedirs(os.path.dirname(save_summary_path), exist_ok=True)
    with open(save_summary_path, "w", encoding="utf-8") as f:
        f.write(content)

    print(f"Summary saved to {save_summary_path}")
    print("\n=== Final Results (One-by-one trim with DOMINANCE + SPLIT slack; All Patients Preserved) ===")
    print(f"Total samples (original):  {total_original:,}")
    print(f"Total samples (final):     {total_final:,}")
    print(f"Patients preserved:        {result_df['session_code'].nunique()} / {df['session_code'].nunique()}")
    print(f"Bottleneck (pre-trim):     {bottleneck:,}")
    print(f"Avg class total:           {avg_class_total:,}")
    print(f"class_slack: {class_slack:.0%}; dominance_cap: {dominance_cap:.0%}; split_slack: {split_slack:.0%}")

    display(summary_df)
    return result_df


# ---- Run splitting ----
num = 1
save_path = f"datasplit/data{num}"
while os.path.exists(save_path):
    print(num)
    num += 1
    save_path = f"datasplit/data{num}"
os.makedirs(save_path)

df = pd.read_csv("audio_datasets/dataset_all_audio.csv")
meta = pd.read_csv("audio_datasets/dataset_meta.csv")

result= split_dataset_by_patient(
    meta,
    save_path=save_path,
    target_class_column="label",
    train_ratio=0.8,
    valid_ratio=0.1,
    test_ratio=0.1,
    max_per_patient=20,
    class_slack=0.50,
    dominance_cap=1.35,
    split_slack=0.10,
    min_allowed_per_patient=1,
    random_state=42,
)


# Check global per-source presence
print(result.pivot_table(index="source", columns="split",
                            values="assigned_count", aggfunc="sum", fill_value=0))

# Check per-class per-source
print(result.groupby(["label","source","split"])["assigned_count"]
                 .sum().unstack("split", fill_value=0))

summary = summarize_splits_with_total(result)
display(summary)

new_df = []
for _, row in result.iterrows():
    assigned_count = row["assigned_count"]
    session_code = row["session_code"]
    split = row["split"]
    df_session = df[df["session_code"]==session_code].sample(assigned_count, random_state=42)
    df_session["split"] = split
    new_df.append(df_session)

new_df = pd.concat(new_df)
df_save_path = os.path.join(save_path,"all_audio_split.csv")
new_df.to_csv(df_save_path,index=False)



# Training/Finetuning

In [ ]:
import os
from pathlib import Path
from datetime import datetime
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchaudio
import matplotlib.pyplot as plt
from visualisation_code.save_visualisations import save_visualisations
from datasets import Dataset, DatasetDict, Value

from transformers import (
    AutoFeatureExtractor,
    AutoModelForAudioClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

from sklearn.metrics import (
    accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report, f1_score, precision_score, recall_score
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class RegressionTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = nn.MSELoss()
        loss = loss_fct(logits.squeeze(), labels)
        return (loss, outputs) if return_outputs else loss

def run_model_on_datasplit(model_ckpt, datasplit, unfreeze_last_n_layers):
    """
    Fine-tunes an audio model using a REGRESSION approach.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    print(f"Running in REGRESSION mode")

    datasplit_folder = f"datasplit/{datasplit}"
    DATA_CSV = os.path.join(datasplit_folder, "all_audio_split.csv")
    df = pd.read_csv(DATA_CSV)

    required_cols = {"speaker_id","audio_name","label","path","session_code","source","split"}
    if not required_cols.issubset(df.columns):
        raise ValueError(f"Missing required columns: {required_cols - set(df.columns)}")

    classes = sorted(df["label"].astype(int).unique().tolist())

    # HF datasets
    def to_ds(split_name):
        sub = df[df["split"] == split_name].reset_index(drop=True).copy()
        sub["labels"] = sub["label"].astype("float32")
        return Dataset.from_pandas(sub)

    dataset_dict = DatasetDict({s: to_ds(s) for s in ["train","valid","test"]})

    # Feature extractor
    feature_extractor = AutoFeatureExtractor.from_pretrained(model_ckpt)
    target_sr = feature_extractor.sampling_rate

    def preprocess_function(examples): 
        paths = examples["path"]
        wavs = []
        for p in paths:
            waveform, sr = torchaudio.load(p)
            if sr != target_sr:
                waveform = torchaudio.transforms.Resample(sr, target_sr)(waveform)
            waveform = waveform.mean(dim=0)
            wavs.append(waveform.numpy())

        inputs = feature_extractor(
            wavs,
            sampling_rate=target_sr,
            truncation=True,
            padding="max_length", # pads shorter audios with silence so all clips are same length (30s)
            max_length = int(target_sr * 30)
        )
        inputs["labels"] = examples["labels"]
        return inputs

    encoded_ds = dataset_dict.map(
        preprocess_function,
        batched=True,
        remove_columns=["speaker_id","audio_name","label","path","session_code","source","split"],
    )

    # Model
    model = AutoModelForAudioClassification.from_pretrained(
        model_ckpt,
        num_labels=1,
    ).to(device)

    # Freeze all parameters first
    for param in model.parameters():
        param.requires_grad = False
    print("All model parameters frozen initially.")
    
    # Identify the correct list of transformer layers based on the model type
    transformer_layers = None
    model_type = ""
    if hasattr(model, 'hubert'):
        transformer_layers = model.hubert.encoder.layers
        model_type = "HuBERT"
    elif hasattr(model, 'wav2vec2'):
        transformer_layers = model.wav2vec2.encoder.layers
        model_type = "Wav2Vec2"
    elif hasattr(model, 'whisper'):
        # Whisper's structure is slightly different (no intermediate model name)
        transformer_layers = model.encoder.layers
        model_type = "Whisper"
    
    # Unfreeze the specified layers
    if transformer_layers:
        if unfreeze_last_n_layers == -1:
            # Unfreeze all layers in the backbone
            for layer in transformer_layers:
                for param in layer.parameters():
                    param.requires_grad = True
            print(f"Unfreezing the ENTIRE {model_type} backbone for full fine-tuning.")
        elif unfreeze_last_n_layers > 0:
            # Unfreeze only the last N layers
            layers_to_unfreeze = transformer_layers[-unfreeze_last_n_layers:]
            for layer in layers_to_unfreeze:
                for param in layer.parameters():
                    param.requires_grad = True
            print(f"Unfreezing the LAST {len(layers_to_unfreeze)} {model_type} transformer layers.")
        else: # unfreeze_last_n_layers == 0
            print(f"Keeping the {model_type} backbone FROZEN.")
    else:
        print("Warning: Could not find a recognizable transformer backbone (hubert, wav2vec2, whisper). Only the head will be trained.")

    # Always unfreeze the classifier head and projector
    if hasattr(model, 'classifier'):
        for param in model.classifier.parameters():
            param.requires_grad = True
        print("Unfreezing the classification head.")
    if hasattr(model, 'projector'):
        for param in model.projector.parameters():
            param.requires_grad = True
        print("Unfreezing the projector.")


    def compute_metrics(pred):
        preds = np.round(pred.predictions).flatten()
        labels = pred.label_ids
        acc = accuracy_score(labels, preds)
        return {"accuracy": acc}

    ts = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    run_name = f"{datasplit}_{Path(model_ckpt).name}_{ts}_regression"
    base_out = Path("models") / run_name
    base_out.mkdir(parents=True, exist_ok=True)

    BATCH = 8
    args = TrainingArguments(
        output_dir=str(base_out / "checkpoints"),
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        learning_rate=5e-5,
        per_device_train_batch_size=BATCH,
        per_device_eval_batch_size=BATCH,
        num_train_epochs=2,
        gradient_accumulation_steps=2,
        warmup_ratio=0.1,
        weight_decay=0.001,
        logging_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_accuracy",
        greater_is_better=True,
        fp16=True,
        report_to=[],
    )

    trainer = RegressionTrainer(
        model=model,
        args=args,
        train_dataset=encoded_ds["train"].with_format("torch"),
        eval_dataset=encoded_ds["valid"].with_format("torch"),
        tokenizer=feature_extractor,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=20, early_stopping_threshold=0)],
    )

    # -------- Train --------
    trainer.train()

    # Save best model + extractor
    best_dir = base_out / "best"
    trainer.model.save_pretrained(best_dir)
    feature_extractor.save_pretrained(best_dir)

    # -------- Evaluate on test --------
    test_ds = encoded_ds["test"].with_format("torch")
    test_metrics = trainer.evaluate(test_ds)
    print("\n--- Test Set Metrics ---")
    print(test_metrics)

    pred_out = trainer.predict(test_ds)
    y_pred = np.round(pred_out.predictions).flatten()
    y_true = pred_out.label_ids

    df_test = df[df["split"] == "test"].reset_index(drop=True).copy()
    df_test["pred"] = y_pred
    df_test["true"] = y_true
    df_test_path = base_out / "test_predictions.csv"
    df_test.to_csv(df_test_path, index=False)
    save_visualisations(df_test,base_out,trainer)



datasplit = "data1"
model_ckpt = "openai/whisper-base"
UNFREEZE_LAYERS = 1 # Unfreeze the last n layers: use -1 to unfreeze entire model

run_model_on_datasplit(model_ckpt, datasplit, unfreeze_last_n_layers=UNFREEZE_LAYERS)

# Inference

In [ ]:
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification
import torch
import librosa
import numpy as np

# Load model and preprocessor
MODEL_PATH = "whisper_base_model"  # Update with your model path
processor = AutoFeatureExtractor.from_pretrained(MODEL_PATH)
model = AutoModelForAudioClassification.from_pretrained(MODEL_PATH)
model.eval()  # set model to evaluation mode

# Load audio file, convert to model inputs
audio_path = "../audio_datasets/Youtube/diarization outputs filtered consolidated/audio 11/speaker 4 moderate/segments _=2s/segment 3.wav"
y, sr = librosa.load(audio_path, sr=processor.sampling_rate)

# Preprocess audio - truncate/pad to 30 seconds 
max_length = int(processor.sampling_rate * 30)
model_inputs = processor(
    y, 
    sampling_rate=processor.sampling_rate, 
    return_tensors="pt", 
    max_length=max_length, 
    truncation=True, 
    padding="max_length"
)

# Inference
with torch.no_grad():
    outputs = model(**model_inputs)
    logits = outputs.logits
    
    # For regression task, gets raw prediction value
    prediction = logits.squeeze().cpu().numpy()
    # Round to get final label 
    sample_prediction = np.round(prediction).astype(int)

# Severity mapping
severity_dict = {
    "0": "Normal/Mild",
    "1": "Moderate",
    "2": "Severe",
    "3": "Profound"
}

severity = severity_dict[str(sample_prediction)]
# print(f"Raw prediction: {prediction}")
# print(f"Rounded prediction: {sample_prediction}")
print(f"Predicted label: {severity}")